# 第 7 章 保存记忆：明天接着聊

## 关掉程序就失忆

现在的学习助手能记住对话，也会压缩记忆了。但它的记忆只存在于程序运行的时候：程序一关，或者 Jupyter 一重启，所有记忆就都没了。

小明今天复习到一半，明天打开学习助手，希望它还记得昨天聊到哪里。另外，小明有时想问点别的事情，又不想和复习的对话混在一起，最好能像聊天软件那样，同时有好几段对话，随时切换。

这一章要做三件事：

1. **保存**：每聊完一轮，就把记忆保存到文件里。
2. **编号**：每段记忆有一个自己的编号（ID），用来区分不同的对话。
3. **切换**：输入 `/new` 开始一段新的记忆，输入 `#` 加编号，加载以前保存的记忆。

把程序运行时的数据保存下来，下次还能用，这叫作**持久化**（persistence）。

In [ ]:
import json
import os
import uuid
from abc import ABC, abstractmethod
from enum import Enum
from pathlib import Path
from typing import List

import anthropic
from anthropic.types import UserMessageParam, AssistantMessageParam
from openai import OpenAI
from openai.types.chat import ChatCompletionSystemMessageParam, ChatCompletionUserMessageParam, ChatCompletionAssistantMessageParam

In [ ]:
def note(label: str, text: str):
    """Print one step of the agent's work, indented, so it stands apart from questions and answers."""
    lines = str(text).splitlines() or [""]
    print(f"  * {label}: {lines[0]}")
    for line in lines[1:]:
        print(f"    {line}")

In [ ]:
class MsgType(Enum):
    SYSTEM = "System"
    USER = "User"
    ASSISTANT = "Assistant"

## 把消息变成字典

内存里的对象，不能直接写进文件。我们需要先把它变成一种能写进文件的格式。这里用 **JSON**：一种很常用的文本格式，长得和 Python 的字典、列表几乎一样，人能直接看懂，几乎所有编程语言都能读写它。

一条消息保存成 JSON，大概是这个样子：

```json
{"type": "User", "content": "你好，我叫小明，正在学第 2 部分。"}
```

所以，第一步是让每条消息都能变成一个字典。`Message` 多了一个 `to_dict` 方法，返回消息的角色和文字内容：

In [ ]:
class Message(ABC):

    def __init__(self, msg_type: MsgType):
        self.msg_type = msg_type

    @abstractmethod
    def summary(self) -> str:
        pass

    def to_dict(self) -> dict:
        return {"type": self.msg_type.value, "content": self.summary()}

    def __repr__(self) -> str:
        return f"Message({self.msg_type.value}): {self.summary()}"

In [ ]:
class SystemMessage(Message):

    def __init__(self, description: str):
        super().__init__(MsgType.SYSTEM)
        self.description = description

    def summary(self) -> str:
        return self.description

In [ ]:
class UserMessage(Message):

    def __init__(self, request: str):
        super().__init__(MsgType.USER)
        self.request = request

    def summary(self) -> str:
        return self.request

In [ ]:
class AssistantMessage(Message):

    def __init__(self, content: str):
        super().__init__(MsgType.ASSISTANT)
        self.content = content

    def summary(self) -> str:
        return self.content

反过来，从文件里读出来的是字典，还要把它变回消息对象。`message_from_dict` 根据字典里的 `type`，在一张对照表里找到对应的类，再用 `content` 创建这个类的对象。

把对象变成可以保存的格式，叫作**序列化**；反过来，叫作**反序列化**。

In [ ]:
def message_from_dict(data: dict) -> Message:
    classes = {
        MsgType.SYSTEM.value: SystemMessage,
        MsgType.USER.value: UserMessage,
        MsgType.ASSISTANT.value: AssistantMessage,
    }
    return classes[data["type"]](data["content"])

In [ ]:
system_message = SystemMessage("You're a learning assistant for the book 《从大模型到智能体》 (From LLM to Agent). "
                               "You help beginners learn how to build AI agents.")

## 给记忆编号

`Memory` 多了三样东西：

- `id`：记忆的编号。`uuid.uuid4()` 会生成一个几乎不可能重复的随机编号，我们只取前 8 个字符，比如 `3f9a1c2e`，又短又好记。加载以前的记忆时，就用这个编号。
- `to_dict`：把整段记忆变成一个字典，包括编号和所有消息。
- `from_dict`：反过来，用一个字典创建一段记忆。它前面的 `@classmethod` 表示这是一个**类方法**：不需要先有一个对象，直接用 `Memory.from_dict(...)` 就能调用，专门用来“从别的数据创建对象”。

In [ ]:
class Memory:
    SUMMARIZE_SYSTEM_MSG = "You are a conversation summary assistant."
    SUMMARIZE_USER_MSG = "Please condense the following conversation into a brief summary: "
    SUMMARY_PREFIX = "Historical Summary: "

    def __init__(self, provider, max_tokens=1000, max_messages=3, keep_messages=1, memory_id=None):
        self.provider = provider
        self.id = memory_id or uuid.uuid4().hex[:8]
        self.messages: List[Message] = []
        self.max_tokens = max_tokens
        self.max_messages = max_messages
        self.keep_messages = keep_messages

    def add(self, message: Message):
        self.messages.append(message)
        if len(self.messages) > self.max_messages or self.token_count() > self.max_tokens:
            self.summarize()

    def reset(self):
        self.messages = []

    def history(self, messages: List[Message] = None) -> str:
        return "\n".join(
            f"[{msg.msg_type.value}]: {msg.summary()}" for msg in (messages or self.messages)
        )

    def summarize(self):
        if self.keep_messages >= len(self.messages):
            return

        split = len(self.messages) - self.keep_messages
        to_summarize = self.messages[:split]
        to_keep = self.messages[split:]

        request = UserMessage(Memory.SUMMARIZE_USER_MSG + "\n" + self.history(to_summarize))
        summary = self.provider.call(SystemMessage(Memory.SUMMARIZE_SYSTEM_MSG), [request])
        self.messages = [UserMessage(Memory.SUMMARY_PREFIX + summary.summary())] + to_keep
        note("summary", self.history())

    def token_count(self):
        # rough estimate: counts characters, not real tokens
        return sum(len(msg.summary() or "") for msg in self.messages)

    def to_dict(self) -> dict:
        return {"id": self.id, "messages": [msg.to_dict() for msg in self.messages]}

    @classmethod
    def from_dict(cls, data: dict, provider) -> "Memory":
        memory = cls(provider, memory_id=data["id"])
        memory.messages = [message_from_dict(m) for m in data["messages"]]
        return memory

## 记忆的仓库

`MemoryStore` 负责把记忆存进文件、再从文件里读出来。每段记忆存成一个文件，都放在 `memories` 文件夹里，文件名就是记忆的编号，比如 `memories/3f9a1c2e.json`。

- `__init__`：如果 `memories` 文件夹不存在，就创建它。
- `save`：把记忆变成字典，再用 `json.dumps` 写成 JSON 文字。`ensure_ascii=False` 让中文按原样保存，而不是变成一串 `\u` 开头的编码；`indent=2` 让文件有缩进，方便人看。
- `load`：按编号找到文件，读出 JSON，再用 `Memory.from_dict` 还原成记忆。找不到文件就返回 `None`。

In [ ]:
class MemoryStore:

    def __init__(self, root="memories"):
        self.root = Path(root)
        self.root.mkdir(exist_ok=True)

    def save(self, memory: Memory):
        path = self.root / f"{memory.id}.json"
        path.write_text(json.dumps(memory.to_dict(), ensure_ascii=False, indent=2), encoding="utf-8")

    def load(self, memory_id: str, provider) -> Memory:
        path = self.root / f"{memory_id}.json"
        if not path.exists():
            return None
        return Memory.from_dict(json.loads(path.read_text(encoding="utf-8")), provider)

In [ ]:
class LLMProvider(ABC):

    @abstractmethod
    def call(self, system: SystemMessage, messages: List[Message], model=None, max_tokens=4096):
        pass

In [ ]:
class OpenAIProvider(LLMProvider):

    def __init__(self):
        self.client = OpenAI(
            api_key=os.getenv("OPENAI_API_KEY"),
            base_url=None
        )
        self.default_model = "gpt-5.4-mini"

    def call(self, system: SystemMessage, messages: List[Message], model=None, max_tokens=4096):
        if system:
            messages = [system] + messages
        kwargs = dict(
            model=model or self.default_model,
            max_completion_tokens=max_tokens,
            messages=self._message_schema(messages)
        )
        response = self.client.chat.completions.create(**kwargs)
        return self._response_schema(response)

    @staticmethod
    def _message_schema(messages: List[Message]):
        schema = []
        for msg in messages:
            if isinstance(msg, SystemMessage):
                schema.append(ChatCompletionSystemMessageParam(role="system", content=msg.description))
            elif isinstance(msg, UserMessage):
                schema.append(ChatCompletionUserMessageParam(role="user", content=msg.request))
            elif isinstance(msg, AssistantMessage):
                schema.append(ChatCompletionAssistantMessageParam(role="assistant", content=msg.content))
        return schema

    @staticmethod
    def _response_schema(response):
        if response is None:
            return AssistantMessage("")

        message = response.choices[0].message
        return AssistantMessage(message.content or "")

In [ ]:
class ClaudeProvider(LLMProvider):

    def __init__(self):
        self.client = anthropic.Anthropic(
            api_key=os.getenv("ANTHROPIC_API_KEY")
        )
        self.default_model = "claude-haiku-4-5"

    def call(self, system: SystemMessage, messages: List[Message], model=None, max_tokens=4096):
        kwargs = dict(
            model=model or self.default_model,
            max_tokens=max_tokens,
            messages=self._message_schema(messages)
        )
        if system:
            kwargs["system"] = system.summary()
        response = self.client.messages.create(**kwargs)
        return self._response_schema(response)

    @staticmethod
    def _message_schema(messages: List[Message]):
        schema = []
        for msg in messages:
            if isinstance(msg, UserMessage):
                schema.append(UserMessageParam(role="user", content=msg.request))
            elif isinstance(msg, AssistantMessage):
                schema.append(AssistantMessageParam(role="assistant", content=msg.content))
        return schema

    @staticmethod
    def _response_schema(response):
        if response is None:
            return AssistantMessage("")

        text = "".join(block.text for block in response.content if block.type == "text")
        return AssistantMessage(text)

## Agent 学会切换记忆

`Agent` 的变化有三处：

1. 多了一个 `self.store`，也就是记忆的仓库。
2. 每聊完一轮，就调用 `self.store.save` 自动保存。这样即使程序突然关掉，最多也只会丢失正在进行的这一轮。
3. `chat` 多了两个命令：
   - `/new`：创建一段新的记忆，之前的记忆已经保存在文件里了，不会丢。
   - `#` 加编号：从仓库里加载这段记忆。找不到的话，提示一下就好。

`chat` 开始时，会显示当前记忆的编号，方便以后加载。

In [ ]:
class Agent:

    def __init__(self, provider: LLMProvider, system: SystemMessage):
        self.provider = provider
        self.system = system
        self.store = MemoryStore()
        self.memory = Memory(provider)

    def chat(self, questions: List[str] = None, system: SystemMessage = None):
        questions = iter(questions) if questions is not None else None  # preset questions, or None to type
        note("memory", f"#{self.memory.id}")
        while True:
            message = next(questions, "/quit") if questions else input("YOU: ")
            print(f"\nYOU: {message}")
            if message.strip().lower() == "/quit":
                break
            if not message.strip():
                continue

            if message.strip().lower() == "/new":
                self.memory = Memory(self.provider)
                note("new memory", f"#{self.memory.id}")
                continue

            if message.startswith("#"):
                memory = self.store.load(message[1:].strip(), self.provider)
                if memory is None:
                    note("memory", f"{message} not found")
                else:
                    self.memory = memory
                    note(f"loaded memory #{memory.id}", memory.history())
                continue

            user_message = UserMessage(message)
            response = self.provider.call(system or self.system, self.memory.messages + [user_message])
            self.memory.add(user_message)
            self.memory.add(response)
            self.store.save(self.memory)
            print(f"AI: {response.summary()}")

## 试一试：保存和加载

先聊一句，让学习助手记住小明。我们把这段记忆的编号存在 `saved_id` 里，后面要用：

In [ ]:
provider = OpenAIProvider()
agent = Agent(provider, system_message)
agent.chat(["你好，我叫小明，正在学第 2 部分。"])
saved_id = agent.memory.id  # this memory is saved in memories/

现在，`memories` 文件夹里已经多了一个文件。打开看看：

In [ ]:
print(Path(f"memories/{saved_id}.json").read_text(encoding="utf-8"))

接下来：用 `/new` 开始一段新记忆，问学习助手“我叫什么名字”，它应该答不出来；再用 `#` 加编号加载刚才的记忆，再问一遍：

In [ ]:
# start a new memory, then load the saved one again
agent.chat(["/new", "我叫什么名字？", f"#{saved_id}", "我叫什么名字？"])
# or call agent.chat() without questions to type your own

加载之后，学习助手又记起了小明。即使重启 Jupyter，只要记得编号，也能把这段对话找回来。

> 💡 聊天记录是以普通文本保存在电脑上的，任何能打开这个文件夹的人都能看到。真实的应用里，要注意保护好这些数据。

## 动手试试

- 重启 Jupyter，重新运行代码，然后用 `#` 加编号加载之前的记忆，验证它真的被保存下来了。
- 给 `chat` 加一个 `/list` 命令，列出所有保存过的记忆编号。提示：给 `MemoryStore` 加一个方法，用 `self.root.glob("*.json")` 找到所有文件，文件的 `stem` 属性就是去掉 `.json` 后的文件名。
- 第 4 章的练习里，我们把学习助手做成了命令行程序。把这一章的代码也放进去，试试关掉程序、重新打开、再接着聊。

## 小结

- 把程序运行时的数据保存下来，下次还能用，叫作**持久化**。
- 我们用 **JSON** 保存记忆：`to_dict` 把消息和记忆变成字典（序列化），`message_from_dict` 和 `Memory.from_dict` 再把字典变回对象（反序列化）。
- 每段记忆有一个随机生成的**编号**，`MemoryStore` 把它保存为 `memories/编号.json`。
- `Agent` 每轮自动保存，并支持 `/new` 开始新记忆、`#编号` 加载旧记忆。

到这里，学习助手已经有了一副好记性。不过，回答比较长的时候，小明要盯着屏幕等好几秒，才能一下子看到全部内容。下一章，我们让回答像打字一样，一个字一个字地显示出来。